# Draw BEM Layers

In [1]:
import mindmap as mm
import pyvista as pv
import numpy as np

In [2]:
# configuration
sub    = "sub-pat25"
config = mm.io.read_config("../configurations/spike.yml")
sample = "sample20"

# data
data = mm.io.read_dataset(sub, "fwd", "bem", "cset", "eeg_inv", sample=sample)

# features
src     = data["fwd"]["src"]
src_pts = pv.PolyData(mm.utils.get_src_position(src))
iskull  = mm.utils.surf_to_polydata(data["bem"][2]["rr"], data["bem"][2]["tris"])
oskull  = mm.utils.surf_to_polydata(data["bem"][1]["rr"], data["bem"][1]["tris"])
scalp   = mm.utils.surf_to_polydata(data["bem"][0]["rr"], data["bem"][0]["tris"])
signal  = np.abs(data["cset"].data[:,12])
elec    = pv.PolyData(mm.utils.get_ch_positions(data["eeg_inv"]))

In [3]:
def filter_vertices(datapoints, x=[-np.inf, np.inf], y=[-np.inf, np.inf], z=[-np.inf, np.inf]):
    """
    Delete vertices outside x,y,z bounds.

    Parameters
    ----------
    datapoints: pyvista.PolyData
        Mesh or points.

    x, y, z: list(float)
        low-high bound in x, y or z direction.

    Returns
    -------
    mesh: pyvista.PolyData
        Filtered polydata structure
    """
    points = datapoints.points
    
    # create mask for outside vertices
    mask = np.ones(len(points), dtype=bool)
    for i,r in enumerate([x,y,z]):
        m    = np.logical_or(points[:,i] < r[0], points[:,i] > r[1])
        mask = np.logical_and(mask, m)

    # filter
    if datapoints.is_all_triangles:
        datapoints = datapoints.remove_points(mask)[0]
    else:
        datapoints.points = datapoints.points[~mask]

    return datapoints


scalp  = filter_vertices(scalp, x=[0, np.inf], y=[-np.inf, 0], z=[-np.inf, -0.03])
oskull = filter_vertices(oskull, x=[0, np.inf], y=[-np.inf, 0], z=[-np.inf, -0.03])
iskull = filter_vertices(iskull, x=[0, np.inf], y=[-np.inf, 0], z=[-np.inf, -0.035])
elec   = filter_vertices(elec, x=[0, np.inf], y=[-np.inf, 0], z=[-np.inf, -0.03])

In [11]:
#signal = np.zeros(data["fwd"]["nsource"])
signal = np.abs(data["cset"].data[:,12])
test = mm.viz.plot_brain_signal(src, signal=signal, cmap="Reds")

#test.add_mesh(iskull, color="red", opacity=1, smooth_shading=True)
#test.add_mesh(oskull, color="yellow", opacity=1, smooth_shading=True)
#test.add_mesh(scalp, color="salmon", opacity=1, smooth_shading=True)
#test.add_mesh(elec, render_points_as_spheres=True, point_size=10, color="black")
#test.add_mesh(src_pts, color="red", render_points_as_spheres=True, point_size=3)
test.show()

Widget(value='<iframe src="http://localhost:36741/index.html?ui=P_0x72741953eb70_7&reconnect=auto" class="pyvi…

# Plot Distributions

In [148]:
import numpy as np 
import plotly.graph_objects as go

L = np.array([1,0.5])
s_true = np.array([1,0.5])
y = L@s_true

lam  = 1
beta = 3

s1, s2   = np.linspace(-2,2,100), np.linspace(-2,2,100)
s11, s22 = np.meshgrid(s1,s2)

like     = 1/(2*np.pi) * np.exp(-(1/2) * (y-(L[0]*s11+L[1]*s22))**2)

gauss    = 1/(2*np.pi) * np.exp(-(1/2) * (s11**2+s22**2))
l2       = like*gauss

lap      = 1/(2*beta) * np.exp(-np.abs(s11)/beta) * np.exp(-np.abs(s22)/beta)
l1       = like * lap

In [170]:
prb = lap

fig = go.Figure(data=
    go.Contour(
        z=prb,
        x=s1,
        y=s2,
        colorscale="OrRd",
        line_smoothing=0.85,
        contours=dict(
            start=0,
            end=prb.max(),
            size=0.005,
        ),
        line_width=0,
    ),
)
fig.update_layout(autosize=False, width=1000, height=1000, font_size=30)
fig.show()